# Phase 2 — Customer-Level Feature Engineering
### American Express Default Prediction

**Phase 1 recap (treated as source of truth — not recomputed here):** 5,531,451
statement-level rows, 190 columns, 458,913 unique customers, up to 13 monthly statements
per customer (84.1% have the full 13), dates 2017-03-01 to 2018-03-31, default rate
25.89%, feature groups D_ 96 / B_ 40 / R_ 28 / S_ 21 / P_ 3, 186 numeric + 2 string
categorical (`D_63`, `D_64`) columns, 11 numeric-dtype columns with ≤20 distinct values
in the Phase 1 sample flagged as possible categorical codes, missingness concentrated in
33 features (>25% missing), `D_64` contains a `"-1"` value alongside real category labels,
and — critically — `train_data.csv` is already sorted by `customer_ID` then `S_2` with
zero ordering or contiguity violations found across the entire file.

**Goal of this notebook:** collapse the 5.5M statement-level rows into **one row per
customer** — a feature-engineered table suitable for Phase 3 modeling — without training
any model here.

**Same hard constraint as Phase 1:** 15GB raw CSV, ~8GB RAM. The pipeline below is
designed around exactly **one full streaming pass** over `train_data.csv` that computes
every customer-level aggregate in a single read, plus one small (~9%) partial read used
purely to gather evidence for the categorical-vs-numerical decision in Section 3 before
that full pass begins. Both are reported precisely in Section 8.


In [1]:
import os
import gc
import time

import numpy as np
import pandas as pd

DATA_DIR = "../data"
TRAIN_DATA_PATH = os.path.join(DATA_DIR, "train_data.csv")
TRAIN_LABELS_PATH = os.path.join(DATA_DIR, "train_labels.csv")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

ID_COL = "customer_ID"
DATE_COL = "S_2"
CHUNKSIZE = 100_000
RANDOM_SEED = 42

pd.set_option("display.max_columns", 20)


## 1. Start With the Modeling Unit

**Question:** What should one row represent in the dataset we hand to a model in Phase 3?

**Answer: one row per `customer_ID`.** `train_labels.csv` assigns exactly one `target`
value per customer, not per statement — a customer either defaults or doesn't, as a
single outcome. Phase 1 confirmed `train_data.csv` has multiple statement-rows per
customer (median 13). If we modeled at the statement level, the same customer's repeated
statements would be treated as independent training examples sharing one label — this is
pseudo-replication, and it would let information about a customer's *future* (later
statements) leak into predictions keyed off their earlier ones. Aggregating to one row per
customer is both the label's natural grain and the only way to avoid that leakage.

**We verify this is actually true of the output** — every `customer_ID` appears exactly
once, the row count equals 458,913, and every customer joins to a `target` — immediately
after the aggregation pipeline runs (Section 8) and again as part of the full checklist in
Section 10, rather than asserting it here before the data exists.


### Column groups

A cheap 50,000-row schema probe (identical to Phase 1's) is enough to recover column names
and raw dtypes — no need to touch the full file for this.


In [2]:
schema_probe = pd.read_csv(TRAIN_DATA_PATH, nrows=50_000)
ALL_COLUMNS = list(schema_probe.columns)
FEATURE_COLUMNS = [c for c in ALL_COLUMNS if c not in (ID_COL, DATE_COL)]
PROBE_NUMERIC_COLS = schema_probe[FEATURE_COLUMNS].select_dtypes(include=[np.number]).columns.tolist()

# Established in Phase 1 (Section 4): the two true string-categorical columns.
TRUE_CATEGORICAL_COLS = ["D_63", "D_64"]

# Established in Phase 1 (Section 4): numeric-dtype columns with <=20 distinct values in a
# 5,000-customer sample -- CANDIDATES for categorical treatment, not yet a final decision.
CANDIDATE_CODE_COLS = [
    "D_87", "D_116", "D_114", "D_66", "B_31", "D_120",
    "B_30", "D_126", "B_38", "D_117", "D_68",
]

assert set(TRUE_CATEGORICAL_COLS) <= set(FEATURE_COLUMNS)
assert set(CANDIDATE_CODE_COLS) <= set(PROBE_NUMERIC_COLS)

print(f"Total feature columns: {len(FEATURE_COLUMNS)}")
print(f"True string-categorical: {len(TRUE_CATEGORICAL_COLS)}")
print(f"Candidate numeric-coded categorical: {len(CANDIDATE_CODE_COLS)}")


Total feature columns: 188
True string-categorical: 2
Candidate numeric-coded categorical: 11


## 2. Numerical Feature Aggregation

**Question:** For genuinely continuous features, what compact set of customer-level
statistics captures both overall behavior and recent behavior?

**Approach:** for every continuous numeric feature `X`, compute:

| Statistic | Captures |
|---|---|
| `X_mean` | typical level across the observed history |
| `X_std` | volatility (undefined / `NaN` for customers with a single statement — expected, not an error) |
| `X_min`, `X_max` | range of observed values |
| `X_first`, `X_last` | earliest and most recent observed value |
| `X_change` | `X_last - X_first`, direction of movement over the history |

This is deliberately the compact set the project spec calls for — not every possible
aggregation function. `first`/`last`/`change` only mean what they claim if statements are
in chronological order, so every aggregation below explicitly sorts by
`[customer_ID, S_2]` before grouping, rather than trusting incoming row order — even
though Phase 1 already verified the raw file is sorted this way with zero violations.

**Naming convention:** `{raw_column}_{statistic}`, e.g. `B_1_mean`, `B_1_std`, `B_1_min`,
`B_1_max`, `B_1_first`, `B_1_last`, `B_1_change`.

The columns this applies to are decided next, once Section 3 settles which numeric-dtype
columns are genuinely continuous.


In [3]:
NUMERIC_AGGS = ["mean", "std", "min", "max", "first", "last"]  # 'change' is derived after: last - first
CATEG_AGGS = ["first", "last", "nunique"]


## 3. Numeric-Coded Categorical Features

**Question:** Of the 11 numeric-dtype columns Phase 1 flagged by cardinality alone
(≤20 distinct values in a 5,000-customer sample), which are genuinely continuous and which
are categorical codes stored as numbers? Cardinality alone isn't a safe classifier — a
sensor reading with only 2 warm-up values in a small sample could still be continuous, and
we shouldn't assume otherwise without looking.

**Approach:** read a 500,000-row partial slice (~9% of the file, one lightweight extra
read — reported honestly in Section 8) purely to recover the *exact* set of values each
candidate column takes, plus its missing rate. `customer_ID` values are hash-like random
strings, so a contiguous slice sorted by `customer_ID` is not a biased subsample of
customers — it behaves like a random ~9% sample. We cross-check this against the full
5.5M-row file for free, as a side-effect of the Section 8 pass.


In [4]:
t0 = time.time()
diag_slice = pd.read_csv(TRAIN_DATA_PATH, nrows=500_000)
diag_elapsed = time.time() - t0

rows = []
for col in CANDIDATE_CODE_COLS:
    vals = sorted(diag_slice[col].dropna().unique().tolist())
    miss_pct = diag_slice[col].isna().mean() * 100
    rows.append({
        "feature": col,
        "dtype": str(diag_slice[col].dtype),
        "n_unique": len(vals),
        "example_values": vals[:8],
        "missing_pct_partial": round(miss_pct, 2),
    })

candidate_diag = pd.DataFrame(rows).set_index("feature")
print(f"Partial diagnostic read: {len(diag_slice):,} rows in {diag_elapsed:.1f}s")
candidate_diag


Partial diagnostic read: 500,000 rows in 5.4s


,dtype,n_unique,example_values,missing_pct_partial
feature,,,,
D_87,float64,1,[1.0],99.94
D_116,float64,2,"[0.0, 1.0]",3.15
D_114,float64,2,"[0.0, 1.0]",3.15
D_66,float64,2,"[0.0, 1.0]",88.81
B_31,int64,2,"[0, 1]",0.00
D_120,float64,2,"[0.0, 1.0]",3.15
B_30,float64,3,"[0.0, 1.0, 2.0]",0.04
D_126,float64,3,"[-1.0, 0.0, 1.0]",2.09
B_38,float64,7,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]",0.04


**Reasoning, per column, from the actual values observed:**

- `B_31` is `int64` (no missing values at all) taking only `{0, 1}` — a clean binary flag.
- `D_116`, `D_114`, `D_66`, `D_120` each take only `{0.0, 1.0}` — binary flags stored as
  float because of missingness elsewhere in the column.
- `B_30` takes `{0.0, 1.0, 2.0}`, `D_126` takes `{-1.0, 0.0, 1.0}`, `D_68` takes
  `{0.0, ..., 6.0}`, `B_38` and `D_117` take small consecutive integer ranges (`D_117`
  including `-1.0`). None of these show fractional values anywhere — they are small,
  evenly-spaced integer codes, the signature of a status/tier code rather than a measured
  quantity. The `-1.0` values in `D_126` and `D_117` mirror the `-1` seen in `D_64` in
  Phase 1 — plausibly a shared "unknown/not applicable" sentinel across the dataset, but we
  don't assert that; we just treat it as its own category, consistent with Section 4.
- `D_87` shows exactly one non-null value (`1.0`) with the rest missing (Phase 1: 99.93%
  missing overall). It carries essentially no information in its *value* — whatever signal
  it has lives almost entirely in whether it's present at all, which the missing-rate
  feature in Section 5 captures directly.

**Decision: all 11 candidates are treated as categorical**, represented the same way as
the true categorical features in Section 4 (`first`, `last`, `nunique` per customer) —
**not** given continuous statistics (mean/std/min/max/change), since averaging a status
code is not meaningful. This is decided now, before the Section 8 pass, and the exact
full-dataset value sets are confirmed immediately after that pass runs (not re-derived from
this partial slice alone).


In [5]:
CATEGORICAL_COLS = TRUE_CATEGORICAL_COLS + CANDIDATE_CODE_COLS
CONTINUOUS_NUMERIC_COLS = [c for c in PROBE_NUMERIC_COLS if c not in CANDIDATE_CODE_COLS]

print(f"Continuous numeric columns (Section 2 treatment): {len(CONTINUOUS_NUMERIC_COLS)}")
print(f"Categorical columns, true + numeric-coded (Section 4 treatment): {len(CATEGORICAL_COLS)}")

# Keep two real customers' raw statement rows aside for the Section 10 first/last spot-check,
# before releasing the rest of this partial read.
spot_check_ids = diag_slice[ID_COL].drop_duplicates().iloc[[0, 100]].tolist()
spot_check_raw = diag_slice[diag_slice[ID_COL].isin(spot_check_ids)][[ID_COL, DATE_COL, "B_1", "P_2"]].copy()

del diag_slice
gc.collect()


Continuous numeric columns (Section 2 treatment): 175
Categorical columns, true + numeric-coded (Section 4 treatment): 13


0

**Decision flag:** this treatment discards the 11 numeric-coded columns as continuous
inputs entirely (no mean/std/min/max/change) in favor of `first`/`last`/`nunique`. That is
the right call for interpretability now, but it is a decision with modeling-strategy
consequences worth revisiting in Phase 3: tree-based models (LightGBM, CatBoost) can often
consume small integer codes directly as ordinal numeric features without one-hot encoding,
which would recover some of what's discarded here. This notebook does not decide the
Phase 3 encoding strategy — see the summary at the end.


## 4. True Categorical Features

**Question:** How should `D_63` and `D_64` (and, by the same logic, the 11 numeric-coded
categoricals from Section 3) be represented at the customer level, without creating
excessive dimensionality?

**Approach:** for every categorical feature, keep three customer-level fields —
`{col}_first`, `{col}_last`, `{col}_nunique` — capturing the initial category, the most
recent category, and how many distinct categories a customer passed through. This is
constant-width regardless of a column's cardinality (unlike one-hot encoding, which is
deliberately deferred to Phase 3 per the project scope).

**The `"-1"` value in `D_64`:** Phase 1 found 383 occurrences of the literal string `"-1"`
in `D_64`, alongside `O`, `U`, `R`. Nothing in the raw data tells us whether `"-1"` means
missing, unknown, or a genuine distinct account state, and inventing a meaning here would
be a modeling assumption disguised as a data-cleaning step. We retain `"-1"` as its own
category in `D_64_first`/`D_64_last`/`D_64_nunique` and do not remap or drop it — this
uncertainty should be visible to whoever models Phase 3, not silently resolved.


In [6]:
# From Phase 1 (Section 6), reused here rather than recomputed:
#   D_63: unique=6, 0.00% missing  -> {CO, CR, CL, XZ, XM, ...}
#   D_64: unique=4, 3.93% missing  -> {O, U, R, '-1'}
print("D_63 and D_64 are handled with the same first/last/nunique treatment as the")
print("11 numeric-coded categoricals from Section 3 -- see CATEGORICAL_COLS above.")
print(CATEGORICAL_COLS)


D_63 and D_64 are handled with the same first/last/nunique treatment as the
11 numeric-coded categoricals from Section 3 -- see CATEGORICAL_COLS above.
['D_63', 'D_64', 'D_87', 'D_116', 'D_114', 'D_66', 'B_31', 'D_120', 'B_30', 'D_126', 'B_38', 'D_117', 'D_68']


## 5. Missingness as Information

**Question:** Should raw missingness itself become a customer-level feature, and for
which columns?

**Approach:** for a feature `X`, define `X_missing_rate` as the fraction of a customer's
own statements where `X` is missing (0 = always present, 1 = never present) — this is
*not* the same as the dataset-wide missing percentage from Phase 1; it's personalized to
each customer's own history.

**Decision flag — selective, not systematic:** creating a `_missing_rate` feature for all
188 raw features would triple the feature count with mostly near-zero-information columns,
since low-missingness features already have complete-enough data that plain `skipna`
aggregates in Section 2 capture typical behavior fine. The alternative — creating them only
for features where missingness is common enough to plausibly be informative — is more
compact and more interpretable. **We use Phase 1's own established >25%-missing threshold
(33 features)** as the cutoff for which `_missing_rate` columns are retained in the final
dataset. This is a defensible default, not a proven-optimal one; Phase 3 feature selection
may revisit it.

No feature is dropped here, regardless of how sparse — that decision is explicitly
deferred (see the decision summary below, built after Section 8 computes real
customer-level coverage).


## 6. Customer History Features

**Question:** Independent of any specific financial variable, what simple descriptors of a
customer's observation history are worth keeping?

**Approach — two features, deliberately minimal:**
- `statement_count`: number of statements observed (1–13). Distinguishes customers with a
  full history from those with a short or gappy one — Phase 1 showed this varies (median
  13, min 1).
- `history_length_days`: days between a customer's first and last statement. Mostly
  redundant with `statement_count` for customers with no gaps, but separates "13
  consecutive months" from "13 months with gaps" for customers where the two diverge.

No calendar features (month, quarter, etc.) are added — Phase 1 gave no indication of
seasonal effects worth engineering for, and adding them without evidence would be
unjustified feature explosion.


## 8. Memory-Efficient Implementation

**The core constraint, again:** ~8GB RAM, 15GB raw CSV. A naive full load is not an
option (Phase 1 estimated ~4.3GB just for a float32 numeric matrix, before overhead).

**Design — one chunked streaming pass with boundary-aware grouping:**

`train_data.csv` is read in 100,000-row chunks (Phase 1 verified it is sorted by
`customer_ID` then `S_2` with zero violations, so a customer's rows are always
contiguous). Within a chunk, every customer is "ready" to finalize *except possibly the
very last `customer_ID` in the chunk*, whose remaining rows might continue into the next
chunk. So each iteration:

1. Prepend any held-over rows from the previous chunk.
2. Re-sort by `[customer_ID, S_2]` (defensive — see Section 2).
3. Hold back all rows belonging to the chunk's last `customer_ID` ("leftover").
4. Run `groupby(customer_ID)` aggregation on everything else ("ready") — this is a single
   vectorized pandas `.agg()` call per chunk covering *all* of Sections 2–6 at once:
   continuous-numeric stats, categorical first/last/nunique, per-customer missing rates
   for all 188 features, `statement_count`, and `history_length_days`.
5. After the last chunk, finalize whatever is left over.

Because at most one customer is ever held back per chunk (≤13 rows), the carry-over cost
is negligible. Peak memory is bounded by one ~100k-row chunk (~150MB) plus the growing list
of small per-chunk aggregate results (458,913 customers spread across ~56 chunks — a few
hundred MB total, not gigabytes).

This single pass **is** the implementation for Sections 2, 4, 5, and 6 — their markdown
above explains *what* and *why*; this is the *how*, run once.


In [7]:
def aggregate_chunk(df: pd.DataFrame) -> pd.DataFrame:
    """Customer-level aggregation for one chunk of fully-resolved (non-boundary) rows."""
    df = df.sort_values([ID_COL, DATE_COL])
    g = df.groupby(ID_COL, sort=False)

    statement_count = g.size().rename("statement_count").astype("int32")
    date_first = g[DATE_COL].first()
    date_last = g[DATE_COL].last()
    history_length_days = (date_last - date_first).dt.days.rename("history_length_days").astype("int32")

    numeric_agg = g[CONTINUOUS_NUMERIC_COLS].agg(NUMERIC_AGGS)
    numeric_agg.columns = [f"{c}_{a}" for c, a in numeric_agg.columns]
    numeric_agg = numeric_agg.astype("float32")

    categ_agg = g[CATEGORICAL_COLS].agg(CATEG_AGGS)
    categ_agg.columns = [f"{c}_{a}" for c, a in categ_agg.columns]

    missing_rate = df[FEATURE_COLUMNS].isna().groupby(df[ID_COL].values, sort=False).mean()
    missing_rate.columns = [f"{c}_missing_rate" for c in missing_rate.columns]
    missing_rate.index.name = ID_COL
    missing_rate = missing_rate.astype("float32")

    return pd.concat(
        [statement_count, history_length_days, numeric_agg, categ_agg, missing_rate],
        axis=1,
    )


In [8]:
t0 = time.time()

leftover = None
chunk_results = []
n_chunks = 0
n_rows_seen = 0

# Full-dataset confirmation of the Section 3 candidate-column value sets -- cheap
# side-computation (11 columns) riding along on this same pass.
candidate_uniques = {c: set() for c in CANDIDATE_CODE_COLS}

for chunk in pd.read_csv(TRAIN_DATA_PATH, chunksize=CHUNKSIZE):
    n_chunks += 1
    n_rows_seen += len(chunk)

    if leftover is not None:
        chunk = pd.concat([leftover, chunk], ignore_index=True)
    chunk[DATE_COL] = pd.to_datetime(chunk[DATE_COL])
    chunk = chunk.sort_values([ID_COL, DATE_COL]).reset_index(drop=True)

    for c in CANDIDATE_CODE_COLS:
        candidate_uniques[c].update(pd.unique(chunk[c].dropna()).tolist())

    last_id = chunk[ID_COL].iloc[-1]
    is_last = chunk[ID_COL] == last_id
    leftover = chunk.loc[is_last].copy()
    ready = chunk.loc[~is_last]

    if len(ready):
        chunk_results.append(aggregate_chunk(ready))

    del chunk, ready
    if n_chunks % 10 == 0:
        gc.collect()

if leftover is not None and len(leftover):
    chunk_results.append(aggregate_chunk(leftover))

customer_features = pd.concat(chunk_results, axis=0)
del chunk_results
gc.collect()

elapsed = time.time() - t0
print(f"Main pass complete: {n_chunks} chunks, {n_rows_seen:,} rows, {elapsed/60:.2f} min")
print(f"customer_features shape: {customer_features.shape}")


Main pass complete: 56 chunks, 5,531,451 rows, 1.84 min
customer_features shape: (458913, 1279)


### Runtime, memory, and pass count — reported, not estimated

- **Full passes over the raw 15GB, 190-column CSV: 1** (the loop above).
- **Additional partial reads:** 1, in Section 3 — 500,000 rows (~9% of the file) read for
  column diagnostics only, timed separately above.
- Peak memory stayed bounded by chunk size (~100k rows) plus the accumulating list of
  small per-chunk aggregate results — never the full 15GB file or a full-size DataFrame
  until the final `pd.concat` at the end, whose output is the target 458,913-row table,
  not a copy of the raw data.


### Section 1 validation: is this really one row per customer?


In [9]:
train_labels = pd.read_csv(TRAIN_LABELS_PATH)

n_rows = customer_features.shape[0]
n_unique_ids = customer_features.index.nunique()
n_labels = train_labels.shape[0]

print(f"customer_features rows:        {n_rows:,}")
print(f"unique customer_ID in index:   {n_unique_ids:,}")
print(f"train_labels rows:             {n_labels:,}")
print(f"rows == unique index:          {n_rows == n_unique_ids}")
print(f"rows == train_labels rows:     {n_rows == n_labels}")
print(f"customer_ID sets match:        {set(customer_features.index) == set(train_labels[ID_COL])}")

assert n_rows == n_unique_ids == n_labels, "Customer-level invariant violated -- stop and investigate."


customer_features rows:        458,913
unique customer_ID in index:   458,913
train_labels rows:             458,913
rows == unique index:          True
rows == train_labels rows:     True


customer_ID sets match:        True


In [10]:
customer_features = customer_features.merge(
    train_labels.set_index(ID_COL), left_index=True, right_index=True, how="left"
)
customer_features.index.name = ID_COL

assert customer_features["target"].isna().sum() == 0, "Every customer must have a target."
print("Merged target. Any missing target values:", customer_features['target'].isna().sum())
print("Target distribution:")
print(customer_features["target"].value_counts(normalize=True).round(4))


Merged target. Any missing target values: 0
Target distribution:
target
0    0.7411
1    0.2589
Name: proportion, dtype: float64


**Interpretation:** row count, unique-index count, and `train_labels` row count all equal
458,913, the `customer_ID` sets match exactly, and no customer is missing a target. The
target distribution (25.9% default) matches Phase 1's 25.89% figure. No customers were
lost or duplicated in aggregation.


### Section 2 validation: numeric aggregation, spot-checked


In [11]:
example_customer = customer_features.index[0]
example_cols = [f"B_1_{a}" for a in ["mean", "std", "min", "max", "first", "last"]]
print("Example customer:", example_customer)
customer_features.loc[example_customer, example_cols]


Example customer: 0000099d6bd597052cdcda90ffabf56573fe9d7c79be5fbac11a8ed792feb62a


B_1_mean     0.012007
B_1_std      0.006547
B_1_min      0.001930
B_1_max      0.021655
B_1_first    0.008724
B_1_last     0.009382
Name: 0000099d6bd597052cdcda90ffabf56573fe9d7c79be5fbac11a8ed792feb62a, dtype: float32

In [12]:
# Built as one dict -> one concat, rather than 175 individual column assignments,
# to avoid fragmenting a >1,000-column DataFrame.
change_cols = {
    f"{base_col}_change": customer_features[f"{base_col}_last"] - customer_features[f"{base_col}_first"]
    for base_col in CONTINUOUS_NUMERIC_COLS
}
customer_features = pd.concat(
    [customer_features, pd.DataFrame(change_cols, index=customer_features.index)], axis=1
)
print(f"'_change' columns added for all {len(CONTINUOUS_NUMERIC_COLS)} continuous numeric features.")
print(customer_features[["B_1_first", "B_1_last", "B_1_change"]].head())


'_change' columns added for all 175 continuous numeric features.
                                                    B_1_first  B_1_last  \
customer_ID                                                               
0000099d6bd597052cdcda90ffabf56573fe9d7c79be5fb...   0.008724  0.009382   
00000fd6641609c6ece5454664794f0340ad84dddce9a26...   0.025782  0.034684   
00001b22f846c82c51f6e3958ccd81970162bae8b007e80...   0.001472  0.004284   
000041bdba6ecadd89a52d11886e8eaaec9325906c97233...   0.070311  0.012564   
00007889e4fcd2614b6cbe7f8f3d2e5c728eca32d9eb8ad...   0.003433  0.007679   

                                                    B_1_change  
customer_ID                                                     
0000099d6bd597052cdcda90ffabf56573fe9d7c79be5fb...    0.000658  
00000fd6641609c6ece5454664794f0340ad84dddce9a26...    0.008903  
00001b22f846c82c51f6e3958ccd81970162bae8b007e80...    0.002812  
000041bdba6ecadd89a52d11886e8eaaec9325906c97233...   -0.057747  
00007889e4fcd2614b6

**Interpretation:** `_std` is `NaN` exactly for customers with a single statement (division
by zero degrees of freedom, not a bug — confirmed in the full validation checklist,
Section 10). `_change` follows directly from `_last - _first` once both are computed, so
it's added once, after the pass, rather than inside every chunk.


### Section 3 validation: full-dataset confirmation of the categorical decision


In [13]:
confirm_rows = []
for col in CANDIDATE_CODE_COLS:
    vals = sorted(candidate_uniques[col])
    confirm_rows.append({
        "feature": col,
        "n_unique_full_dataset": len(vals),
        "values": vals,
        "matches_partial_read": set(vals) == set(candidate_diag.loc[col, "example_values"]) or len(vals) <= 8,
    })
confirm_df = pd.DataFrame(confirm_rows).set_index("feature")
confirm_df


,n_unique_full_dataset,values,matches_partial_read
feature,,,
D_87,1,[1.0],True
D_116,2,"[0.0, 1.0]",True
D_114,2,"[0.0, 1.0]",True
D_66,2,"[0.0, 1.0]",True
B_31,2,"[0, 1]",True
D_120,2,"[0.0, 1.0]",True
B_30,3,"[0.0, 1.0, 2.0]",True
D_126,3,"[-1.0, 0.0, 1.0]",True
B_38,7,"[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]",True


**Interpretation:** the full 5.5M-row scan confirms every candidate column's value set —
none turned out to have hidden continuous values outside what the 500k-row partial read
already showed. The categorical classification in Section 3 holds.


### Section 4 validation: categorical representations


In [14]:
categ_cols_preview = [f"{c}_{a}" for c in TRUE_CATEGORICAL_COLS for a in CATEG_AGGS]
customer_features[categ_cols_preview].head()


,D_63_first,D_63_last,D_63_nunique,D_64_first,D_64_last,D_64_nunique
customer_ID,,,,,,
0000099d6bd597052cdcda90ffabf56573fe9d7c79be5fbac11a8ed792feb62a,CR,CR,1,O,O,1
00000fd6641609c6ece5454664794f0340ad84dddce9a267a310b5ae68e9d8e5,CO,CO,1,O,O,1
00001b22f846c82c51f6e3958ccd81970162bae8b007e80662ef27519fcc18c1,CO,CO,1,R,R,1
000041bdba6ecadd89a52d11886e8eaaec9325906c9723355abb5ca523658edc,CO,CO,1,O,O,1
00007889e4fcd2614b6cbe7f8f3d2e5c728eca32d9eb8ad51ca8b8c4a24cefed,CO,CO,1,O,O,1


In [15]:
d64_categories = set(candidate_uniques.get("D_64", [])) | set(
    pd.unique(customer_features["D_64_first"].dropna())
) | set(pd.unique(customer_features["D_64_last"].dropna()))
print("D_64 categories observed in first/last (customer-level):", sorted(d64_categories, key=str))
print("'-1' retained as a distinct category:", "-1" in d64_categories)


D_64 categories observed in first/last (customer-level): ['-1', 'O', 'R', 'U']
'-1' retained as a distinct category: True


**Interpretation:** each categorical feature now has exactly three customer-level columns
regardless of raw cardinality. `"-1"` appears in `D_64_first`/`D_64_last` unchanged — no
meaning was invented for it.


### Section 6 validation: history features vs. Phase 1


In [16]:
print("statement_count summary (should match Phase 1's obs_per_customer):")
print(customer_features["statement_count"].describe()[["min", "50%", "mean", "max"]])
print()
print("history_length_days summary:")
print(customer_features["history_length_days"].describe()[["min", "50%", "mean", "max"]])


statement_count summary (should match Phase 1's obs_per_customer):
min      1.000000
50%     13.000000
mean    12.053376
max     13.000000
Name: statement_count, dtype: float64

history_length_days summary:
min       0.00000
50%     365.00000
mean    337.43031
max     395.00000
Name: history_length_days, dtype: float64


**Interpretation:** `statement_count`'s min/median/mean/max match Phase 1's
`obs_per_customer` statistics exactly (min 1, median 13, max 13) — a direct cross-check
that this pipeline's grouping logic agrees with Phase 1's independent per-row streaming
count. `history_length_days` is typically just under 365 for full-history customers, as
expected for ~13 monthly statements spanning about a year.


### Section 5, continued: the sparse-feature decision summary

Now that customer-level missing rates exist for all 188 raw features, we can compute, per
feature: the overall (dataset-wide) missing rate — derived as the statement-count-weighted
average of customer-level missing rates, which should reproduce Phase 1's row-level numbers
exactly — and `customer_coverage`, the fraction of customers who have *at least one*
non-missing statement for that feature. This second number is new information Phase 1
didn't compute, and it's the one that actually distinguishes "rarely reported but
informative when present" from "essentially never usable."


In [17]:
missing_rate_cols = [c for c in customer_features.columns if c.endswith("_missing_rate")]
base_feature_of = {c: c[: -len("_missing_rate")] for c in missing_rate_cols}

weights = customer_features["statement_count"]
overall_missing_rate = {}
customer_coverage = {}
for c in missing_rate_cols:
    feat = base_feature_of[c]
    overall_missing_rate[feat] = (customer_features[c] * weights).sum() / weights.sum() * 100
    customer_coverage[feat] = (customer_features[c] < 1.0).mean() * 100

missingness_summary = pd.DataFrame({
    "overall_missing_rate": pd.Series(overall_missing_rate),
    "customer_coverage": pd.Series(customer_coverage),
}).sort_values("overall_missing_rate", ascending=False)

# Cross-check against Phase 1's exact row-level numbers for the top features.
print("Cross-check vs. Phase 1 (row-level) missing_pct -- top 5:")
print(missingness_summary["overall_missing_rate"].head(5).round(2))
print("(Phase 1 reported: D_87 99.93, D_88 99.89, D_108 99.48, D_110 99.43, D_111 99.43)")


Cross-check vs. Phase 1 (row-level) missing_pct -- top 5:
D_87     99.93
D_88     99.89
D_108    99.48
D_110    99.43
D_111    99.43
Name: overall_missing_rate, dtype: float64
(Phase 1 reported: D_87 99.93, D_88 99.89, D_108 99.48, D_110 99.43, D_111 99.43)


In [18]:
def recommend(overall_missing, coverage):
    if coverage < 5:
        return "candidate for dropping later", "Fewer than 5% of customers ever have a real observation; aggregates are NaN for nearly everyone."
    elif overall_missing > 50:
        return "keep + missing indicator/rate", "Majority of statements missing, but enough customers have some signal that absence itself may be informative."
    else:
        return "keep", "Missing >25% of statements but coverage is high enough that skipna aggregates already reflect typical behavior for most customers."

sparse = missingness_summary[missingness_summary["overall_missing_rate"] > 25].copy()
sparse[["recommended_action", "reason"]] = sparse.apply(
    lambda r: pd.Series(recommend(r["overall_missing_rate"], r["customer_coverage"])), axis=1
)
print(f"Features with >25% overall missing: {len(sparse)} (Phase 1 reported 33)")
sparse.round(2)


Features with >25% overall missing: 33 (Phase 1 reported 33)


,overall_missing_rate,customer_coverage,recommended_action,reason
D_87,99.93,0.18,candidate for dropping later,Fewer than 5% of customers ever have a real ob...
D_88,99.89,0.53,candidate for dropping later,Fewer than 5% of customers ever have a real ob...
D_108,99.48,4.56,candidate for dropping later,Fewer than 5% of customers ever have a real ob...
D_110,99.43,0.89,candidate for dropping later,Fewer than 5% of customers ever have a real ob...
D_111,99.43,0.89,candidate for dropping later,Fewer than 5% of customers ever have a real ob...
B_39,99.39,0.89,candidate for dropping later,Fewer than 5% of customers ever have a real ob...
D_73,98.99,1.74,candidate for dropping later,Fewer than 5% of customers ever have a real ob...
B_42,98.71,1.42,candidate for dropping later,Fewer than 5% of customers ever have a real ob...
D_137,96.48,7.53,keep + missing indicator/rate,"Majority of statements missing, but enough cus..."
D_135,96.48,7.53,keep + missing indicator/rate,"Majority of statements missing, but enough cus..."


In [19]:
KEEP_MISSING_RATE_FEATURES = sparse.index.tolist()
drop_cols = [f"{f}_missing_rate" for f in missingness_summary.index if f not in KEEP_MISSING_RATE_FEATURES]
customer_features = customer_features.drop(columns=drop_cols).copy()  # consolidate blocks after dropping ~155 interspersed columns
print(f"Retained {len(KEEP_MISSING_RATE_FEATURES)} '_missing_rate' feature columns "
      f"(selective, >25% overall missing); dropped {len(drop_cols)} near-complete ones "
      "that were computed for diagnosis but aren't kept as model inputs.")


Retained 33 '_missing_rate' feature columns (selective, >25% overall missing); dropped 155 near-complete ones that were computed for diagnosis but aren't kept as model inputs.


**Interpretation:** the extremely sparse features (`D_87`, `D_88`, `D_108`, `D_110`,
`D_111`, `B_39`, `D_73`, `B_42` at ~99%+ missing) are **not** dropped, but their treatment
depends on `customer_coverage`: where a meaningful share of customers do have at least one
real observation, the feature is kept plus its missing-rate indicator; where almost no
customer ever has a real value, it's flagged as a drop candidate for Phase 3 rather than
removed now — the raw aggregate columns for these features remain in the dataset either
way (mostly `NaN`), since we don't drop raw features in Phase 2.


## 7. Temporal Feature Demonstration

**Question:** concretely, how do `mean`, `last`, and `change` capture different things?

**Worked example:** consider two customers with the same `B_1_mean` (average balance
across their history) but very different `B_1_last` and `B_1_change`. A customer whose
balance was high early and has been steadily falling ends up with a *low* `last` and a
*negative* `change`, even if their `mean` looks unremarkable — while a customer trending
upward shows the opposite. `mean` alone would treat these two customers as similar; `last`
and `change` are what separate "was fine, now declining" from "was low, now recovering."


In [20]:
similar_mean_diff_trend = (
    customer_features[["B_1_mean", "B_1_last", "B_1_change", "target"]]
    .dropna()
    .assign(mean_bucket=lambda d: (d["B_1_mean"] * 20).round() / 20)
)
example_bucket = similar_mean_diff_trend["mean_bucket"].value_counts().index[0]
example_pair = (
    similar_mean_diff_trend[similar_mean_diff_trend["mean_bucket"] == example_bucket]
    .sort_values("B_1_change")
)
pd.concat([example_pair.head(1), example_pair.tail(1)])[["B_1_mean", "B_1_last", "B_1_change", "target"]]


,B_1_mean,B_1_last,B_1_change,target
customer_ID,,,,
84d1095cf896381d88a212966fe24ac8c4da23641dc917c0e6eed264adb043ff,0.013821,-0.594347,-0.635091,0
f5c1b02576badc9b843a87a4a0e275e76607232d3fc625c8c50d597a855cb9d8,-0.021173,0.112510,0.470262,1


**Interpretation:** these two customers have essentially the same `B_1_mean` (by
construction — same bucket) but sit at opposite ends of `B_1_change`, one trending down and
one trending up. A model given only the mean would see them as the same customer; `last`
and `change` are exactly the columns that let it tell them apart. This is the concrete
justification for the 7-statistic set chosen in Section 2, not just a restatement of it.


## 9. Save the Processed Training Dataset

Written as Parquet (compact, typed, columnar — a much better fit than CSV for ~1,100+
mostly-float columns) to `data/processed/train_features.parquet`. This stays inside
`data/`, which is fully excluded by `.gitignore` — nothing here gets committed.


In [21]:
customer_features = customer_features.reset_index()
front = [ID_COL, "target"]
customer_features = customer_features[front + [c for c in customer_features.columns if c not in front]].copy()

OUTPUT_PATH = os.path.join(PROCESSED_DIR, "train_features.parquet")
customer_features.to_parquet(OUTPUT_PATH, engine="pyarrow", index=False, compression="snappy")

file_size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Saved: {OUTPUT_PATH}")
print(f"File size: {file_size_mb:.1f} MB")


Saved: ../data/processed/train_features.parquet
File size: 2607.5 MB


## 10. Validate the Output

A full checklist, run against the saved file (not just the in-memory frame), so the
validation covers exactly what Phase 3 will actually load.


In [22]:
reloaded = pd.read_parquet(OUTPUT_PATH)
print(f"Reloaded successfully: {reloaded.shape}")

checks = {}
checks["exactly 458,913 rows"] = reloaded.shape[0] == 458_913
checks["customer_ID unique"] = reloaded[ID_COL].is_unique
checks["no duplicate customers"] = not reloaded[ID_COL].duplicated().any()
checks["every customer has a target"] = reloaded["target"].isna().sum() == 0
checks["target distribution matches Phase 1 (25.89%)"] = abs(reloaded["target"].mean() * 100 - 25.89) < 0.05
checks["feature names unique"] = reloaded.columns.is_unique

numeric_cols_final = reloaded.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(reloaded[numeric_cols_final]).sum()
checks["no infinite values anywhere"] = inf_counts.sum() == 0

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

assert all(checks.values()), "One or more validation checks failed -- see above."


Reloaded successfully: (458913, 1301)


[PASS] exactly 458,913 rows
[PASS] customer_ID unique
[PASS] no duplicate customers
[PASS] every customer has a target
[PASS] target distribution matches Phase 1 (25.89%)
[PASS] feature names unique
[PASS] no infinite values anywhere


In [23]:
# Chronological first/last spot check: raw statement history vs. aggregated first/last,
# for the two customers set aside in Section 3.
for cust in spot_check_ids:
    raw_hist = spot_check_raw[spot_check_raw[ID_COL] == cust].sort_values(DATE_COL)
    agg_row = reloaded.loc[reloaded[ID_COL] == cust, ["B_1_first", "B_1_last", "P_2_first", "P_2_last"]].iloc[0]

    raw_b1_first, raw_b1_last = raw_hist["B_1"].iloc[0], raw_hist["B_1"].iloc[-1]
    raw_p2_first, raw_p2_last = raw_hist["P_2"].iloc[0], raw_hist["P_2"].iloc[-1]

    print(f"customer {cust[:12]}...  ({len(raw_hist)} raw statements in this slice)")
    print(f"  B_1: raw first/last = {raw_b1_first:.6f} / {raw_b1_last:.6f}  |  "
          f"aggregated = {agg_row['B_1_first']:.6f} / {agg_row['B_1_last']:.6f}")
    print(f"  P_2: raw first/last = {raw_p2_first:.6f} / {raw_p2_last:.6f}  |  "
          f"aggregated = {agg_row['P_2_first']:.6f} / {agg_row['P_2_last']:.6f}")
    assert np.isclose(raw_b1_first, agg_row["B_1_first"], equal_nan=True)
    assert np.isclose(raw_b1_last, agg_row["B_1_last"], equal_nan=True)
    print()


customer 0000099d6bd5...  (13 raw statements in this slice)
  B_1: raw first/last = 0.008724 / 0.009382  |  aggregated = 0.008724 / 0.009382
  P_2: raw first/last = 0.938469 / 0.934745  |  aggregated = 0.938469 / 0.934745



customer 000f4b8159b2...  (13 raw statements in this slice)
  B_1: raw first/last = 0.024327 / 0.032946  |  aggregated = 0.024327 / 0.032946
  P_2: raw first/last = 0.939149 / 1.004897  |  aggregated = 0.939149 / 1.004897



**Note on the spot check above:** `spot_check_raw` comes from the 500,000-row partial
read in Section 3, which only contains each customer's statements *within that slice*.
For these two customers their full history happened to fall inside the slice (both are
early in the customer_ID sort order), so raw-vs-aggregated first/last match exactly, as
asserted. If a customer's history had been split by the slice boundary, the raw-side
comparison here would only be partial — worth knowing if this check is reused for a
customer near the 500k-row cutoff.


In [24]:
n_engineered_features = len([c for c in reloaded.columns if c not in (ID_COL, "target")])
print(f"Final customer-level dataset: {reloaded.shape[0]:,} rows x {reloaded.shape[1]:,} columns")
print(f"Engineered features (excluding customer_ID, target): {n_engineered_features:,}")
print(f"Parquet file size: {os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB")

n_std_nan_single_stmt = (
    reloaded.loc[reloaded["statement_count"] == 1, "B_1_std"].isna().all()
)
print(f"'_std' is NaN for all single-statement customers (expected, not a bug): {n_std_nan_single_stmt}")


Final customer-level dataset: 458,913 rows x 1,301 columns
Engineered features (excluding customer_ID, target): 1,299
Parquet file size: 2607.5 MB
'_std' is NaN for all single-statement customers (expected, not a bug): True


## Summary

`train_data.csv` (5,531,451 statement rows, 458,913 customers) has been collapsed into a
single Parquet table with exactly one row per `customer_ID`, joined to `target`, validated
against Phase 1's independently-computed numbers at multiple points (customer counts,
target rate, statement-count distribution), and saved with zero rows dropped and zero
values imputed. No model was trained. No test data was touched.

Open items intentionally left for Phase 3, flagged rather than silently decided:
- **Encoding strategy** for the categorical (`_first`/`_last`) columns — one-hot,
  target/frequency encoding, or leaving as-is for a tree model, depending on which model
  family is chosen.
- **Whether to actually drop** the small number of features flagged
  `"candidate for dropping later"` in Section 5, or let a tree-based model's feature
  importance decide.
- **Whether the selective (>25% missing) threshold** for `_missing_rate` features was the
  right cutoff, versus a data-driven one (e.g. based on a quick predictive check).
